# Reranker
Use this notebook to configure your retriever pipelines and test them on `train.jsonl` as validation.

Current Reranker Selection:
- [] Qwen/qwen2.5-7b-instruct
- [] Qwen/qwen2.5-32b-instruct

In [1]:
# === IMPORT LIBRARIES === 
import sys
import os
from pathlib import Path
import ctypes
import pandas as pd

In [2]:
# === RESOLVE PATHS ===

# CRITICAL SYSTEM BOOT FIX: Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Add parent directory to python path to import from src/
sys.path.append(str(Path("..").resolve()))

In [3]:
# === IMPORT MODULES ===
from src.data.loader import load_jsonl
from src.evaluation.metrics import evaluate_retriever
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.dense import DenseRetriever
from src.retrievers.llm import LLMRetriever
from src.retrievers.cross_encoder import CrossEncoderRetriever

# === IMPORT HF API TOKEN ===
env_path = Path("..") / ".env"
if env_path.exists():
    with open(env_path, "r") as f:
        for line in f:
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.strip().split("=", 1)
                os.environ[k.strip()] = v.strip()
    print("✅ Successfully Authenticated Hugging Face Token!")


✅ Successfully Authenticated Hugging Face Token!


In [4]:
# === IMPORT VALIDATION DATA ===
DATA_DIR = Path("../data")
val_data = load_jsonl(str(DATA_DIR / "train.jsonl"))
print(f"Loaded {len(val_data)} validation samples.")

Loaded 2055 validation samples.


In [5]:
# === EXPERIMENT CONFIGURATION ===
CONFIG = {
    "retriever": "llm",
    "dense_model_name": "BAAI/bge-m3", 
    "llm_model_name": "Qwen/Qwen2.5-32B-Instruct",
    "cross_encoder_model_name": "BAAI/bge-reranker-v2-m3", 
    "dense_scores_path": "../outputs/cache/ensemble_dense_scores/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_train.json", 
    "top_k": 5,
    "max_candidates": 12, 
    "max_chars": 1500 # (Add this to prevent OOM on the 32B model)
}
os.environ["DISABLE_VLM"] = "1"
os.environ["VLM_CAPTIONS_FILE"] = "vlm_image_captions/qwen2-vl-7b-instruct_image_captions.json"


In [6]:
# === RETRIEVER SELECTION ===
if CONFIG["retriever"] == "bm25":
    retriever = BM25Retriever()
elif CONFIG["retriever"] == "dense":
    retriever = DenseRetriever(
        model_name=CONFIG["dense_model_name"]
    )
elif CONFIG["retriever"] == "llm":
    retriever = LLMRetriever(
        model_name=CONFIG["llm_model_name"], 
        load_in_4bit=True, 
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
elif CONFIG["retriever"] == "cross_encoder":
    retriever = CrossEncoderRetriever(
        model_name=CONFIG["llm_model_name"],
        load_in_4bit=True,
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
else:
    raise NotImplementedError(f"Retriever type '{CONFIG['retriever']}' not recognized.")

Loading pre-computed dense scores from ../outputs/cache/ensemble_dense_scores/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_train.json...
Initializing LLM Retriever with model 'Qwen/Qwen2.5-32B-Instruct' (4-bit=True, disable_filtering=False, max_candidates=12, max_chars=1500, hybrid_mode=True)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

In [7]:
# === RUN VALIDATION ===
from pathlib import Path

# Set the new flag to True so it returns the DataFrame too!
recall, df_val = evaluate_retriever(retriever, val_data, k=CONFIG["top_k"], return_predictions=True)
print(f"\nValidation Recall@{CONFIG['top_k']}: {recall:.4f}")

🏃‍♂️ Starting validation loop across 2054 valid samples...


Pipeline Evaluation (Recall@5):   0%|          | 0/2054 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# === SAVE VALIDATION PREDICTIONS ===
output_dir = Path("../outputs/validation")
output_dir.mkdir(parents=True, exist_ok=True)

# Build the filename dynamically exactly like we do in the submission script
mode = "hybrid" if CONFIG.get("dense_scores_path") else "vanilla"
model_stub = CONFIG[CONFIG['retriever']+'_model_name'].split('/')[-1].lower()
val_name = f"validation_{CONFIG['retriever']}_{model_stub}-{mode}.csv" if CONFIG["retriever"] != "bm25" else "validation_bm25-baseline.csv"

In [ ]:
output_path = output_dir / val_name
df_val.to_csv(output_path, index=False)
print(f"\n✅ Successfully saved validation predictions to: {output_path}")